# M1 — Semantic Scholar references (Colab + Drive)

Chạy **sau** notebook crawl arXiv của người 1 (`M1_Colab_Concurrent_Limited.ipynb`).
Quy trình: GitHub branch → Drive → đọc ID từ checkpoint `batch_*.tar.gz` → crawl references (resume) → xuất 2 bảng `papers` / `references` + validate.

- Mỗi paper có trạng thái: `ok` / `not_indexed` / `failed` / `invalid_id`.
- `not_indexed`: Semantic Scholar trả về HTTP 404.
- `failed`: lỗi HTTP, lỗi mạng, lỗi xử lý hoặc ghi dữ liệu.
- Paper có 0 references vẫn được ghi thành file JSON rỗng `[]`, không bị crawl lại.
- Resume:
  - File JSON hợp lệ (`ok`) được bỏ qua.
  - `not_indexed` và `failed` được thử lại ở lần chạy tiếp theo.
  - File JSON bị hỏng sẽ được crawl lại.
  - `invalid_id` không được gửi đến API.
- Lỗi từng paper không dừng cả run.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-Update-pipeline"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m1_pipeline.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive của range

`RANGE_ID` và `DATA_DIR` phải trùng với notebook crawl arXiv của người 1 để đọc được checkpoint.

**Lưu ý:** Notebook này chỉ đọc checkpoint của người 1, không ghi references hoặc tables vào `DATA_DIR`. Dữ liệu Semantic Scholar được lưu trong thư mục riêng của bạn.

Cấu trúc Google Drive:

```text
MyDrive/
└── references/
    └── range_4/               # MY_REFS_DIR - dữ liệu của bạn
        ├── reference/         # Toàn bộ references đã crawl
        │   ├── 2307-00001.json
        │   ├── 2307-00002.json
        │   └── ...
        │
        └── tables/            # Các bảng và báo cáo
            ├── papers.json
            ├── references.json
            ├── metrics_s2_references.json
            ├── s2_crawl_records.csv
            └── validation_report.json

In [ ]:
RANGE_ID = 4

In [ ]:

import os
from datetime import datetime, timezone
from pathlib import Path

from google.colab import userdata

# 1. DATA_DIR: checkpoint của Người 1 (chỉ đọc)
DATA_DIR = Path(
    f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev"
)

# 2. Thư mục riêng trên Drive để lưu kết quả S2
MY_REFS_DIR = Path(
    f"/content/drive/MyDrive/references/range_{RANGE_ID}"
)

REFS_ROOT = MY_REFS_DIR / "reference"
TABLES_DIR = MY_REFS_DIR / "tables"

for path in (MY_REFS_DIR, REFS_ROOT, TABLES_DIR):
    path.mkdir(parents=True, exist_ok=True)

# 3. Kiểm tra checkpoint
manifests = sorted(DATA_DIR.glob("batch_*.json"))

assert manifests, (
    f"Chưa tìm thấy batch_*.json trong {DATA_DIR} "
    "— kiểm tra lại lối tắt Drive"
)

print(f"Range: {RANGE_ID}")
print(f"Đọc checkpoint từ: {DATA_DIR}")
print(f"Số batch tìm thấy: {len(manifests)}")
print(f"Lưu references tại: {REFS_ROOT}")
print(f"Lưu tables tại: {TABLES_DIR}")

# 4. Lấy API key từ environment hoặc Colab Secrets
S2_API_KEY_DIRECT = os.environ.get("S2_API_KEY", "")

if not S2_API_KEY_DIRECT:
    try:
        S2_API_KEY_DIRECT = userdata.get("S2_API_KEY")
    except Exception:
        S2_API_KEY_DIRECT = ""

if S2_API_KEY_DIRECT:
    os.environ["S2_API_KEY"] = S2_API_KEY_DIRECT
    print("Đã tìm thấy Semantic Scholar API key.")
else:
    print("Chưa tìm thấy API key.")

# 5. Run ID
RUN_ID = os.environ.get(
    "S2_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
print("S2_RUN_ID =", RUN_ID)

# 6. Cấu hình đường dẫn cho pipeline và metrics
from src.common import config
import src.common.metrics as metrics_module

config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

print("\nTABLES_DIR =", config.TABLES_DIR)
print("REFS_ROOT =", REFS_ROOT)
print("S2_MIN_DELAY_SEC =", config.S2_MIN_DELAY_SEC)
print("S2_MIN_DELAY_UNAUTH_SEC =", config.S2_MIN_DELAY_UNAUTH_SEC)
print("S2_429_COOLDOWN_SEC =", config.S2_429_COOLDOWN_SEC)


## 2 — Đọc danh sách paper từ checkpoint `batch_*.tar.gz`

Mỗi archive chứa `papers.jsonl` (metadata người 1 đã crawl). Lấy metadata của **mọi** bài, không lọc theo trạng thái tải TeX: bài chỉ có PDF vẫn là một hàng của bảng `papers` và vẫn có references. Bài S2/arXiv không tìm thấy (`not_found`) sẽ chỉ được báo cáo, không làm dừng notebook.

In [ ]:
import json
from pathlib import Path
import tarfile

from src.common.ids import normalize_arxiv_id
from src.m1_crawling.assigned_ranges import assigned_ids

# Đảm bảo các biến đường dẫn đã được khai báo trước đó (DATA_DIR, TABLES_DIR, RANGE_ID)
OUTPUT_TABLES_DIR = TABLES_DIR
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def _read_jsonl_from_archive(
    archive_path: Path,
    filename: str
) -> list[dict]:
    """Đọc JSONL trong archive .tar.gz mà không giải nén ra đĩa."""
    records = []

    with tarfile.open(archive_path, "r:gz") as tar:
        try:
            member = tar.extractfile(filename)
        except KeyError:
            member = None

        if member is None:
            raise ValueError(
                f"{archive_path.name} thiếu file {filename}"
            )

        with member:
            for line in member:
                line = line.strip()
                if line:
                    records.append(
                        json.loads(line.decode("utf-8"))
                    )

    return records


def load_all_papers_from_all_batches() -> list[dict]:
    papers_dict = {}
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Range {RANGE_ID} | Tìm thấy {len(manifests)} batch trong {DATA_DIR}")
    for index, manifest in enumerate(manifests, start=1):
        archive = manifest.with_suffix(".tar.gz")
        if not archive.exists():
            print(f"[warn] Bỏ qua (thiếu .tar.gz): {archive.name}")
            continue
        try:
            rows = _read_jsonl_from_archive(archive, "papers.jsonl")
        except (tarfile.TarError, EOFError, OSError, ValueError) as exc:
            print(f"[warn] Batch lỗi, cần crawl lại: {archive.name} ({exc})")
            continue
        for row in rows:
            try:
                paper_id = normalize_arxiv_id(row.get("arxiv_id") or "")
            except ValueError:
                print(
                    f"[warn] Bỏ qua dòng có ID không hợp lệ: {row.get('arxiv_id')!r}"
                )
                continue
            papers_dict[paper_id] = {
                "arxiv_id": paper_id,
                "title": row.get("title", ""),
                "authors": row.get("authors", []),
                "submission_date": row.get("submission_date", ""),
                "revised_dates": row.get("revised_dates", []),
                "venue": row.get("venue", None),
            }
        if index % 10 == 0 or index == len(manifests):
            print(
                f"  -> {index}/{len(manifests)} batch (tích lũy {len(papers_dict)} bài)"
            )
    return sorted(papers_dict.values(), key=lambda p: p["arxiv_id"])


def save_papers_to_json(papers_meta: list[dict], output_dir: Path):
    """Lưu danh sách metadata bài báo ra file papers.json."""
    output_dir.mkdir(parents=True, exist_ok=True)

    json_path = output_dir / "papers.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(
            papers_meta,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"✓ Đã lưu file JSON: {json_path}")


# --- Thực thi ---
papers_meta = load_all_papers_from_all_batches()
print(
    f"\nHOÀN THÀNH RANGE {RANGE_ID}: {len(papers_meta)} bài báo có metadata."
)

# Đối chiếu với danh sách ID đề giao (chỉ báo cáo, không assert)
expected = set(assigned_ids(RANGE_ID))
got = {p["arxiv_id"] for p in papers_meta}
missing = sorted(expected - got)
print(
    f"Đề giao: {len(expected)} | Có metadata: {len(got & expected)} | "
    f"Thiếu: {len(missing)} | Thừa ngoài range: {len(got - expected)}"
)
if missing:
    print(
        "[warn] Chưa có metadata (batch chưa crawl, lỗi, hoặc ID không tồn tại trên arXiv):",
        missing[:10],
        "..." if len(missing) > 10 else "",
    )
    print(
        "      Xem reports/crawl_records.csv của người 1 để biết status (error / not_found)."
    )

# Lưu dữ liệu ra file papers.json
save_papers_to_json(papers_meta, OUTPUT_TABLES_DIR)

## 3 — Crawl references (resume, có đo runtime)

Mỗi paper được lưu thành một file JSON riêng trong `REFS_ROOT/`.
Tên file được tạo bằng `folder_name_for_arxiv_id()`.

Ví dụ:
- arXiv ID: `2307.00001`
- File: `2307-00001.json`

File chứa toàn bộ references của paper, bao gồm cả reference không có arXiv ID.

In [ ]:
import csv
import logging
import os
import time

from src.common.io import write_json
from src.common.metrics import directory_size_bytes, track_run
from src.m1_crawling.s2_batch import DiskPeakSampler, run_s2_batch, summarize_s2_results

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s"
)

if "papers_meta" not in globals() or not papers_meta:
    raise RuntimeError(
        "'papers_meta' chưa được tạo! Hãy chạy lại cell ở Mục 2 trước."
    )

api_key = globals().get("S2_API_KEY_DIRECT") or os.environ.get("S2_API_KEY")


def _progress(index, total, result):
    if index % 50 == 0 or index == total:
        print(f"  -> Progress: {index}/{total} papers processed...")


started_at = time.perf_counter()

with track_run("s2_references") as metrics:
    metrics["range_id"] = RANGE_ID
    metrics["run_id"] = RUN_ID
    metrics["s2_min_delay_sec"] = config.S2_MIN_DELAY_SEC
    metrics["s2_min_delay_unauth_sec"] = config.S2_MIN_DELAY_UNAUTH_SEC
    metrics["s2_429_cooldown_sec"] = config.S2_429_COOLDOWN_SEC
    metrics["n_ids"] = len(papers_meta)

    disk_sampler = DiskPeakSampler(REFS_ROOT).start()
    crawl_started_at = time.perf_counter()
    results, n_invalid_ids = run_s2_batch(
        papers_meta=papers_meta,
        refs_root=REFS_ROOT,
        api_key=api_key,
        progress=_progress,
    )
    crawl_elapsed = time.perf_counter() - crawl_started_at
    disk_sampler.stop()
    cell_elapsed = time.perf_counter() - started_at

    stats = summarize_s2_results(results)
    stats["n_invalid_ids"] = n_invalid_ids
    stats["crawl_elapsed_sec"] = crawl_elapsed
    stats["cell_elapsed_sec"] = cell_elapsed
    stats["disk"] = disk_sampler.result()
    metrics["s2_stats"] = stats
    metrics["measurement_scope"] = {
        "elapsed_s": (
            "one fresh publication API wait including pacing/cooldown and "
            "retries; excludes normalization and cache write; cache hits "
            "use elapsed_s='' and are not measured as 0"
        ),
        "crawl_elapsed_sec": (
            "run_s2_batch loop: cache reads, API requests, normalization, "
            "cache writes"
        ),
        "cell_elapsed_sec": (
            "this cell up to before metrics/CSV export; excludes checkpoint "
            "loading and validation/table export"
        ),
        "n_ok": "includes cache hits",
        "success_rate_all_input": "n_ok / n_input, cache hits included",
        "success_rate_fresh": (
            "n_ok_fresh / n_attempted_fresh; excludes cache hits"
        ),
        "n_http_requests": "HTTP attempts actually made, retries included",
        "n_http_retries": "retry attempts after the first attempt",
        "n_http_429": "responses with status 429, including retried ones",
        "n_api_errors": "HTTP/network errors only",
        "n_processing_errors": (
            "invalid JSON/payload, local processing/write errors, rejected "
            "requests"
        ),
        "avg_s2_seconds_per_publication_fetch": (
            "fresh API publications incl. retries/pacing; denominator "
            "n_attempted_fresh"
        ),
        "rss": (
            "current Python process only; child processes excluded; "
            "kernel/Colab baseline included"
        ),
    }
    metrics["not_measured"] = [
        "historical full-crawl wall time: lượt hiện tại có thể phần lớn là cache, "
        "không thay cho benchmark crawl mới toàn bộ",
    ]

write_json(TABLES_DIR / "metrics_s2_references.json", metrics)

records_csv = TABLES_DIR / "s2_crawl_records.csv"
fields = [
    "range_id",
    "run_id",
    "arxiv_id",
    "status",
    "n_refs",
    "n_arxiv",
    "n_refs_without_arxiv",
    "is_zero_ref",
    "is_api_error",
    "is_invalid_id",
    "from_cache",
    "elapsed_s",
    "n_invalid_refs",
    "n_http_requests",
    "n_http_retries",
    "n_http_429",
    "error_kind",
    "error",
]

with records_csv.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for record in results:
        row = {key: record.get(key, "") for key in fields}
        row["range_id"] = RANGE_ID
        row["run_id"] = RUN_ID
        writer.writerow(row)

stats["output_bytes"] = {
    "references_dir_bytes": directory_size_bytes(REFS_ROOT),
    "tables_dir_bytes": directory_size_bytes(TABLES_DIR),
    "size_unit": "bytes",
    "note": (
        "final size after this cell's metrics/CSV export; uncompressed "
        "JSON/CSV; not peak disk; peak disk during the crawl is "
        "stats['disk']['peak_disk_bytes']; only these sibling directories "
        "are counted; validation/table export is added later; the metrics "
        "file is rewritten once after measurement so its byte count may "
        "differ by a few bytes"
    ),
}
metrics["s2_stats"] = stats
write_json(TABLES_DIR / "metrics_s2_references.json", metrics)

print("\n✓ ĐÃ XUẤT THÀNH CÔNG VÀ LƯU BẢNG CSV:")
print(f" - n_invalid_ids : {n_invalid_ids}")
print(f" - n_zero_refs   : {stats['n_zero_refs']}")
print(f" - n_api_errors  : {stats['n_api_errors']}")
print(f" - n_processing_errors : {stats['n_processing_errors']}")
print(f" - n_cache_hits  : {stats['n_cache_hits']}")
print(f" - HTTP requests : {stats['n_http_requests']}")
print(f" - HTTP retries  : {stats['n_http_retries']} "
      f"({stats['retry_rate']:.3%})")
print(f" - HTTP 429      : {stats['n_http_429']}")
print(f" - Peak disk     : {stats['disk']['peak_disk_bytes']} bytes")
print(f" - File CSV tại  : {records_csv}")



## 4 — Recovery: chạy lại phần references dở dang
- Một lần tra cứu Semantic Scholar cho mỗi publication trong điều kiện bình thường. Nếu request gặp lỗi, `request_with_retry` có thể thực hiện thêm lần thử.
- Mỗi paper có trạng thái: `ok`, `not_indexed`, `failed`, `invalid_id`.
- `not_indexed`: Semantic Scholar trả về HTTP 404.
- `failed`: lỗi HTTP, lỗi mạng, lỗi xử lý hoặc ghi dữ liệu.
- Paper có 0 references vẫn được lưu thành file JSON rỗng `[]`, được xem là hoàn thành.

- Resume:
  - File JSON hợp lệ (`ok`) được bỏ qua.
  - `not_indexed` và `failed` được thử lại ở lần chạy tiếp theo.
  - File JSON bị hỏng sẽ được crawl lại.
  - `invalid_id` không được gửi đến API.

In [ ]:
from src.m1_crawling.s2_batch import find_empty_cache_ids, run_s2_batch

# Cache rỗng có thể hợp lệ (bài 0 references) hoặc do code cũ coi thiếu
# field "references" là []. Rà danh sách rồi chỉ recrawl các ID cần kiểm tra.
empty_cache_ids = find_empty_cache_ids(papers_meta, REFS_ROOT)
print(f"Cache rỗng: {len(empty_cache_ids)} bài")
if empty_cache_ids:
    print("Ví dụ:", empty_cache_ids[:20])

S2_RECRAWL_IDS: list[str] = []  # ví dụ ["2310.00001"]
if S2_RECRAWL_IDS:
    recovery_api_key = (
        globals().get("api_key")
        or globals().get("S2_API_KEY_DIRECT")
        or os.environ.get("S2_API_KEY")
    )
    results, n_invalid_ids = run_s2_batch(
        papers_meta,
        REFS_ROOT,
        api_key=recovery_api_key,
        force_recrawl_ids=S2_RECRAWL_IDS,
    )
    print(f"Đã recrawl {len(S2_RECRAWL_IDS)} bài; invalid_id = {n_invalid_ids}")

## 5 — Xuất 2 bảng `papers` / `references` + validate

- `papers`: metadata đọc từ checkpoint của người 1, một hàng cho mỗi publication.
- `references`: chỉ chứa reference có arXiv ID, phục vụ matching pool của M3.
- Toàn bộ references, kể cả reference không có arXiv ID, được lưu trong các file JSON riêng tại `REFS_ROOT/`.
- Tên file được tạo bằng `folder_name_for_arxiv_id()`, ví dụ `2307-00001.json`.

Kết quả validate (thiếu thông tin / sai kiểu / trùng lặp) được in ra và ghi vào `validation_report.json`.

In [ ]:
import json
from pathlib import Path

from src.common.io import write_json

from src.common import config
from src.common.ids import folder_name_for_arxiv_id
from src.common.schemas import PaperRow, ReferenceRow
from src.m1_crawling.tables import (
    dedupe_reference_rows,
    s2_records_to_reference_rows,
    validate_m1_tables,
    write_m1_tables,
)

# Đảm bảo trỏ đường dẫn xuất file về đúng Drive
config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR

paper_rows: list[PaperRow] = []
reference_rows: list[ReferenceRow] = []

# 1. Gom dữ liệu papers và references từ các file JSON phẳng
for row in papers_meta:  # papers_meta đã unique theo arxiv_id
    base = row["arxiv_id"]

    # Tạo đối tượng PaperRow
    paper_rows.append(
        PaperRow(
            arxiv_id=base,
            title=row.get("title", ""),
            authors=row.get("authors") or [],
            submission_date=row.get("submission_date", ""),
            revised_dates=row.get("revised_dates") or [],
            venue=row.get("venue"),
        )
    )

    # Trỏ trực tiếp đến file JSON phẳng (ví dụ: 2307-00001.json)
    safe_filename = f"{folder_name_for_arxiv_id(base)}.json"
    target = REFS_ROOT / safe_filename

    if target.exists():
        try:
            with open(target, "r", encoding="utf-8") as f:
                raw_records = json.load(f)
            # s2_records_to_reference_rows đã tự động dedup bên trong
            reference_rows.extend(
                s2_records_to_reference_rows(base, raw_records)
            )
        except Exception as exc:
            print(f"[warn] Không thể đọc file references {target}: {exc}")

# 2. Loại bỏ trùng lặp toàn bộ danh sách dựa trên bộ 3 (arxiv_id, ref_arxiv_id, ref_s2_id)
n_before = len(reference_rows)
reference_rows = dedupe_reference_rows(reference_rows)
print(
    f"references: {n_before} -> {len(reference_rows)} sau khi bỏ trùng toàn bộ bộ 3 (arxiv_id, ref_arxiv_id, ref_s2_id)"
)

# 3. Validate dữ liệu 2 bảng (ĐÃ SỬA LỖI KEYERROR)
report = validate_m1_tables(paper_rows, reference_rows)
report["range_id"] = RANGE_ID
report["run_id"] = RUN_ID
report["dedupe_before"] = n_before
report["dedupe_after"] = len(reference_rows)
report["generated_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(
    "validate ok =",
    report["ok"],
    "| papers =",
    report["n_papers"],
    "| references =",
    report["n_references"],
    "| issues =",
    report["n_issues"],
    "| warnings =",
    report["n_warnings"],
)

# In tối đa 20 issues đầu tiên nếu có lỗi/cảnh báo
for issue in report["issues"][:20]:
    print("  [issue]", issue)
for warning in report["warnings"][:20]:
    print("  [warning]", warning)

# 4. Ghi dữ liệu ra file papers.json và references.json
write_m1_tables(paper_rows, reference_rows)
write_json(TABLES_DIR / "validation_report.json", report)

# 5. Thống kê kết quả xuất file trên Drive
n_ref_files = len(list(REFS_ROOT.glob("*.json")))
print(f"\nTổng số file JSON phẳng đã cào trong REFS_ROOT: {n_ref_files} bài")

sample = next(REFS_ROOT.glob("*.json"), None)
if sample:
    with open(sample, "r", encoding="utf-8") as f:
        sample_rows = json.load(f)
    with_arxiv = sum(1 for r in sample_rows if r.get("ref_arxiv_id"))
    print(
        f"Ví dụ mẫu ({sample.name}): {len(sample_rows)} refs, {with_arxiv} có arXiv ID"
    )
    print("Một record mẫu:", sample_rows[0] if sample_rows else None)

print("\nĐÃ XUẤT THÀNH CÔNG:")
for name in (
    "papers.json",
    "references.json",
    "s2_crawl_records.csv",
    "metrics_s2_references.json",
    "validation_report.json",
):
    path = TABLES_DIR / name
    print(
        f"- {name}: {path.stat().st_size} bytes"
        if path.exists()
        else f"- {name}: MISSING"
    )

## Sau khi chạy

- Gửi cho người xử lý dữ liệu:
  - `tables/papers.json`
  - `tables/references.json`
  - `tables/s2_crawl_records.csv`
  - Toàn bộ thư mục `references/range_4/reference/`.

- Thư mục `reference/` chứa toàn bộ references, bao gồm cả ref không có arXiv ID. Không xóa thư mục này trước khi hoàn thành M3 và submit Hugging Face.